In [ ]:
#| default_exp handlers.general

# General Handler

`encode(yaml_path)` — marisco 2.0 のグランドマスター関数。
YAML パス 1 つを渡すだけで、全 3 フェーズ（loader → transformer → writer）を直線的に駆動し、
生 CSV/TSV を MARIS 正典 NetCDF4 へ自動整流・製本する。

In [ ]:
#| export
from __future__ import annotations
from pathlib import Path
from marisco.handlers.pipeline.callbacks import PipelineState
from marisco.handlers.pipeline.assembly import (
    run_finalize as _run_finalize,
    run_preflight as _run_preflight,
)
from marisco.handlers.pipeline.contracts import HandlerConfig, PluginSpec
from marisco.handlers.pipeline.gates import gap_check
from marisco.handlers.pipeline.intake import call_loader, resolve_loader_fn
from marisco.handlers.pipeline.writer  import write_netcdf


In [ ]:
#| export
def _load_plugin_fn(spec: PluginSpec, yaml_dir: Path = None):
    "Fail-Fast dynamic import of a plain function (not instantiated); for custom loaders."
    return resolve_loader_fn(spec, yaml_dir)


def _call_loader(cfg: HandlerConfig, yaml_dir: Path = None):
    "Resolve the configured loader and execute it with cfg plus any declared loader args."
    return call_loader(cfg, yaml_dir=yaml_dir)


def _load_cfg(yaml_path: str | Path, fname_out: str = None):
    "Bind YAML to HandlerConfig and apply an optional output override."
    yaml_path = Path(yaml_path)
    yaml_dir = yaml_path.parent
    cfg = HandlerConfig.from_yaml(yaml_path)
    gap_check(cfg)
    if fname_out:
        cfg = cfg.model_copy(update={"fname_out": fname_out})
    return cfg, yaml_dir


def _init_state(cfg: HandlerConfig, yaml_dir: Path = None) -> PipelineState:
    "Load provider data and wrap it in PipelineState."
    return PipelineState(dfs=_call_loader(cfg, yaml_dir=yaml_dir))


def _verify_success_message(cfg: HandlerConfig, state: PipelineState) -> str:
    lines = [
        f"Congratulations - Gate 1 and Gate 2 both passed for {cfg.title or cfg.module_name!r}.",
        "The declarative pipeline is physically consistent through the pre-lossy checkpoint.",
    ]
    for grp, df in state.dfs.items():
        lines.append(f"- {grp}: {len(df):,} rows x {df.shape[1]} cols after declarative canonicalization")
    return "\n".join(lines)


def verify(yaml_path: str | Path, fname_out: str = None) -> PipelineState:
    "Dry-run a YAML-configured dataset through Gate 1, declarative transforms, and deep Gate 2 without writing NetCDF."
    cfg, yaml_dir = _load_cfg(yaml_path, fname_out=fname_out)
    state = _init_state(cfg, yaml_dir=yaml_dir)
    _run_preflight(state, cfg, yaml_dir=yaml_dir)
    print(_verify_success_message(cfg, state))
    return state


def encode(yaml_path: str | Path, fname_out: str = None) -> None:
    "Encode any YAML-configured dataset to MARIS NetCDF4 in a pure, phase-aware pipeline."
    cfg, yaml_dir = _load_cfg(yaml_path, fname_out=fname_out)
    state = _init_state(cfg, yaml_dir=yaml_dir)
    _run_preflight(state, cfg, yaml_dir=yaml_dir)
    _run_finalize(state, cfg, yaml_dir=yaml_dir)
    write_netcdf(state, cfg)


## encode

| フェーズ | 呼び出し | 役割 |
|--------|---------|------|
| Phase 0 | `HandlerConfig.from_yaml` + `gap_check` | 型安全バインド ＆ 必須列の事前検疫 |
| Phase 1 | `load_data` | 生 CSV/TSV → `{grp: DataFrame}` |
| Phase 2 | `run_preflight` + `run_finalize` | 宣言的正規化 + 標準コアレイル + 出力前ガード |
| Phase 3 | `write_netcdf` | 2フェーズ Strict ガード → `.nc` 物理書き出し |

YAML は純宣言層であり、実行ロジックは loader と共通エンジンに限定される。

In [ ]:
def _load_plugin_fn(spec: PluginSpec, yaml_dir: Path = None):
    "Fail-Fast dynamic import of a plain function (not instantiated); for custom loaders."
    return resolve_loader_fn(spec, yaml_dir)


def _call_loader(cfg: HandlerConfig, yaml_dir: Path = None):
    "Resolve the configured loader and execute it with cfg plus any declared loader args."
    return call_loader(cfg, yaml_dir=yaml_dir)


def _load_cfg(yaml_path: str | Path, fname_out: str = None):
    "Bind YAML to HandlerConfig and apply an optional output override."
    yaml_path = Path(yaml_path)
    yaml_dir = yaml_path.parent
    cfg = HandlerConfig.from_yaml(yaml_path)
    gap_check(cfg)
    if fname_out:
        cfg = cfg.model_copy(update={"fname_out": fname_out})
    return cfg, yaml_dir


def _init_state(cfg: HandlerConfig, yaml_dir: Path = None) -> PipelineState:
    "Load provider data and wrap it in PipelineState."
    return PipelineState(dfs=_call_loader(cfg, yaml_dir=yaml_dir))


def _verify_success_message(cfg: HandlerConfig, state: PipelineState) -> str:
    lines = [
        f"Congratulations - Gate 1 and Gate 2 both passed for {cfg.title or cfg.module_name!r}.",
        "The declarative pipeline is physically consistent through the pre-lossy checkpoint.",
    ]
    for grp, df in state.dfs.items():
        lines.append(f"- {grp}: {len(df):,} rows x {df.shape[1]} cols after declarative canonicalization")
    return "\n".join(lines)


def verify(yaml_path: str | Path, fname_out: str = None) -> PipelineState:
    "Dry-run a YAML-configured dataset through Gate 1, declarative transforms, and deep Gate 2 without writing NetCDF."
    cfg, yaml_dir = _load_cfg(yaml_path, fname_out=fname_out)
    state = _init_state(cfg, yaml_dir=yaml_dir)
    _run_preflight(state, cfg, yaml_dir=yaml_dir)
    print(_verify_success_message(cfg, state))
    return state


def encode(yaml_path: str | Path, fname_out: str = None) -> None:
    "Encode any YAML-configured dataset to MARIS NetCDF4 in a pure, phase-aware pipeline."
    cfg, yaml_dir = _load_cfg(yaml_path, fname_out=fname_out)
    state = _init_state(cfg, yaml_dir=yaml_dir)
    _run_preflight(state, cfg, yaml_dir=yaml_dir)
    _run_finalize(state, cfg, yaml_dir=yaml_dir)
    write_netcdf(state, cfg)


In [ ]:
import sys
sys.stdout.reconfigure(encoding="utf-8")
from pathlib import Path
from marisco.handlers.pipeline.loader import HandlerConfig, PluginSpec, gap_check
from marisco.handlers.general import encode, _load_plugin, build_core_pipeline, resolve_callback

cfg = HandlerConfig.from_yaml("config/handlers/fram_strait.yaml")
gap_check(cfg)

core = build_core_pipeline(cfg)
print(f"build_core_pipeline (FramStrait, 1 unit_conv): {len(core)} CBs")
for i, cb in enumerate(core, 1):
    print(f"  {i:2d}. {type(cb).__name__}")

# Topology guard smoke test: empty dfs → guards degrade to Null-Object
from marisco.callbacks import Transformer
import pandas as pd
tfm_empty = Transformer({'SEAWATER': pd.DataFrame()}, cbs=core)
tfm_empty()  # must not raise KeyError for TIME/LON/LAT absent
print("\nTopology guard ✓ — no KeyError on empty DataFrame (Null-Object auto-degrade)")

# ── resolve_callback: legacy path ──
spec_path = PluginSpec(path="marisco.callbacks.shared.SoftRegexTransformCB")
cls = resolve_callback(spec_path)
from marisco.callbacks.shared import SoftRegexTransformCB
assert cls is SoftRegexTransformCB, "legacy path resolution failed"
print("resolve_callback(path) ✓")

# ── resolve_callback: name shorthand (shared scan) ──
spec_name = PluginSpec(name="SoftDMStoDecimalCB")
cls = resolve_callback(spec_name)
from marisco.callbacks.shared import SoftDMStoDecimalCB
assert cls is SoftDMStoDecimalCB, "name shorthand resolution failed"
print("resolve_callback(name) ✓")

# ── resolve_callback: local file ──
import tempfile, textwrap
dummy_src = textwrap.dedent("""
    from marisco.callbacks.core import PerGroupCB
    class DummyLocalCB(PerGroupCB):
        grps = ['SEAWATER']
        def each_grp(self, grp, df, tfm): pass
""")
with tempfile.TemporaryDirectory() as tmpdir:
    local_py = Path(tmpdir) / "local_cbs.py"
    local_py.write_text(dummy_src)
    spec_file = PluginSpec(**{"file": "local_cbs.py", "class": "DummyLocalCB"})
    cls = resolve_callback(spec_file, yaml_dir=Path(tmpdir))
    assert cls.__name__ == "DummyLocalCB"
    instance = _load_plugin(spec_file, yaml_dir=Path(tmpdir))
    assert type(instance).__name__ == "DummyLocalCB"
print("resolve_callback(file+class) ✓")

# ── Fail-Fast: unknown name raises ImportError ──
try:
    resolve_callback(PluginSpec(name="NonExistentCB"))
    assert False, "Should have raised"
except ImportError:
    pass
print("Fail-Fast unknown name ✓")

# ── Fail-Fast: bad path raises immediately ──
try:
    _load_plugin(PluginSpec(path="marisco.nonexistent.FakeCB"))
except (ImportError, ModuleNotFoundError) as e:
    print(f"Fail-Fast bad path ✓ — {type(e).__name__}")

# normalize_case smoke test
cfg_norm = cfg.model_copy(update={"normalize_case": {"nuclide_raw": "NUCLIDE"}})
from marisco.callbacks import LowerStripNameCB
norm_cbs = [LowerStripNameCB(col_src=s, col_dst=d) for s, d in cfg_norm.normalize_case.items()]
assert len(norm_cbs) == 1 and type(norm_cbs[0]).__name__ == "LowerStripNameCB"
print("normalize_case → LowerStripNameCB ✓")

In [ ]:
#|eval: false
# Full pipeline integration test (requires network access to Zenodo)
encode("config/handlers/fram_strait.yaml", fname_out="_data/output/FramStrait_general.nc")
print("FramStrait_general.nc written.")